In [6]:
!hdfs dfs -mv /user/mahasiswa/tugas5/transaksi_tugas5.csv /user/haris/tugas5/

In [5]:
!hdfs dfs -mkdir -p /user/haris/tugas5

0. Inisialisasi SparkSession & Preparation

In [14]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, count, row_number
from pyspark.sql.window import Window
import pandas as pd

# 1. Inisialisasi SparkSession
spark = SparkSession.builder \
    .appName("TugasMandiri5") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# 2. Baca file CSV dari HDFS dengan awalan hdfs://
df_transaksi = spark.read.csv(
    "hdfs://localhost:9000/user/haris/tugas5/transaksi_tugas5.csv", 
    header=True, 
    inferSchema=True
)

# 3. Tambahkan kolom pendapatan (unit_terjual * harga_satuan)
df_transaksi = df_transaksi.withColumn("pendapatan", col("unit_terjual") * col("harga_satuan"))

# 4. Buat df_target dari dictionary data_target_cabang
data_target_cabang = {
    "kota": ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"],
    "target_bulanan": [45000000, 60000000, 55000000, 40000000, 30000000],
    "pic_cabang": ["Rani", "Joko", "Sari", "Bayu", "Fitri"],
}
df_target = spark.createDataFrame(pd.DataFrame(data_target_cabang))

print("df_transaksi dan df_target berhasil dibuat!")

df_transaksi dan df_target berhasil dibuat!


A. Join & Perbandingan Target (Bobot 25%)

In [15]:
# Meringkas total pendapatan per kota
df_ringkasan_kota = df_transaksi.groupBy("kota").agg(
    spark_sum("pendapatan").alias("total_pendapatan")
)

# Join dengan df_target
df_bagian_a = df_ringkasan_kota.join(df_target, on="kota", how="inner") \
    .withColumn("pencapaian_persen", (col("total_pendapatan") / col("target_bulanan")) * 100) \
    .orderBy(col("pencapaian_persen").desc())

df_bagian_a.show()

[Stage 5:>                                                          (0 + 8) / 8]

+----------+----------------+--------------+----------+------------------+
|      kota|total_pendapatan|target_bulanan|pic_cabang| pencapaian_persen|
+----------+----------------+--------------+----------+------------------+
| Purworejo|        45650000|      30000000|     Fitri|152.16666666666669|
|      Solo|        33475000|      40000000|      Bayu|           83.6875|
|Yogyakarta|        47275000|      60000000|      Joko| 78.79166666666667|
|  Magelang|        31650000|      45000000|      Rani| 70.33333333333334|
|  Semarang|        38175000|      55000000|      Sari|  69.4090909090909|
+----------+----------------+--------------+----------+------------------+



B. Window Function — Kategori Terlaris per Kota (Bobot 25%)

In [16]:
# Agregasi total pendapatan berdasarkan pasangan kota & kategori
df_kategori_kota = df_transaksi.groupBy("kota", "kategori").agg(
    spark_sum("pendapatan").alias("total_pendapatan_kategori")
)

# Definisi window: kelompokkan berdasarkan kota, urutkan dari pendapatan tertinggi
window_spec_kota = Window.partitionBy("kota").orderBy(col("total_pendapatan_kategori").desc())

# Tambahkan row_number dan filter top-1
df_bagian_b = df_kategori_kota.withColumn("rank", row_number().over(window_spec_kota)) \
    .filter(col("rank") == 1) \
    .drop("rank") \
    .orderBy("kota")

df_bagian_b.show()

+----------+--------------------+-------------------------+
|      kota|            kategori|total_pendapatan_kategori|
+----------+--------------------+-------------------------+
|  Magelang|Kesehatan & Kecan...|                  7275000|
| Purworejo|Kesehatan & Kecan...|                 10075000|
|  Semarang|        Rumah Tangga|                 11125000|
|      Solo|Kesehatan & Kecan...|                  8425000|
|Yogyakarta|             Fashion|                 13325000|
+----------+--------------------+-------------------------+



C. Spark SQL (Bobot 25%)

In [17]:
# Registrasi temporary view
df_transaksi.createOrReplaceTempView("transaksi")
df_target.createOrReplaceTempView("target")

# Kueri Spark SQL
df_bagian_c = spark.sql('''
    SELECT 
        t.kota, 
        tg.pic_cabang, 
        COUNT(t.order_id) AS jumlah_transaksi
    FROM transaksi t
    JOIN target tg ON t.kota = tg.kota
    GROUP BY t.kota, tg.pic_cabang
    ORDER BY jumlah_transaksi DESC
''')

df_bagian_c.show()

+----------+----------+----------------+
|      kota|pic_cabang|jumlah_transaksi|
+----------+----------+----------------+
| Purworejo|     Fitri|             116|
|Yogyakarta|      Joko|             110|
|      Solo|      Bayu|              95|
|  Semarang|      Sari|              93|
|  Magelang|      Rani|              86|
+----------+----------+----------------+



D. Kesimpulan (Bobot 25%)

Cabang Performa Terbaik (Purworejo - PIC: Fitri) Cabang Purworejo mencatatkan kinerja paling unggul dengan melampaui target bulanan hingga 152,17% (pendapatan Rp45.650.000 dari target Rp30.000.000). Capaian ini didukung oleh volume penjualan tertinggi sebanyak 116 transaksi, dengan pendorong utama berasal dari kategori Kesehatan & Kecantikan yang menyumbang Rp10.075.000.

Cabang yang Membutuhkan Perhatian (Semarang & Magelang) Cabang Semarang (PIC: Sari) dan Magelang (PIC: Rani) memerlukan intervensi langsung dari manajemen. Semarang menjadi cabang dengan persentase pencapaian terendah, yakni 69,41% (pendapatan Rp38.175.000 dari target Rp55.000.000). Cabang Magelang mencatat pencapaian 70,33% (pendapatan Rp31.650.000 dari target Rp45.000.000) dan menjadi cabang dengan jumlah transaksi paling sedikit (86 transaksi). Sementara itu, Cabang Yogyakarta mencetak pendapatan nominal tertinggi (Rp47.275.000), namun persentase pencapaiannya baru menyentuh 78,79% karena beban target yang tinggi (Rp60.000.000).

Rekomendasi Strategis Manajemen disarankan mengevaluasi ulang penetapan target untuk Semarang dan Magelang, atau menyalurkan bantuan dorongan promosi yang lebih agresif. Strategi optimalisasi penjualan kategori Kesehatan & Kecantikan yang berhasil di Purworejo, Solo, dan Magelang juga dapat direplikasi ke Cabang Semarang guna mendongkrak omzet yang saat ini masih didominasi kategori Rumah Tangga.